# 09b · Cierre formal de modelos en dev

## Objetivo
Cerrar formalmente la selección del mejor modelo en `dev` con una rúbrica multicriterio y generar artefactos de freeze de modelo.

## Rol en la metodología
Este notebook separa la etapa de **decisión/freeze** de la etapa de **comparación de resultados**.

## Entradas
- `data/outputs/barridos_hibridos/<timestamp>/tabla_maestra_comparativa.csv`
- `data/outputs/barridos_hibridos/<timestamp>/ranking_variantes.csv`
- `data/outputs/freeze_lexico_<timestamp>/freeze_lexico_resumen.json`

## Salidas
- `data/outputs/cierre_modelos_dev_<timestamp>/ranking_modelos_dev.csv`
- `data/outputs/cierre_modelos_dev_<timestamp>/rubrica_seleccion_modelos.csv`
- `data/outputs/cierre_modelos_dev_<timestamp>/decision_modelo_final.md`
- `data/outputs/cierre_modelos_dev_<timestamp>/decision_modelo_final.json`
- `data/outputs/cierre_modelos_dev_<timestamp>/lista_modelos_para_test.json`
- `data/outputs/cierre_modelos_dev_<timestamp>/riesgos_y_limitaciones_dev.md`

## Notebook anterior
- `notebooks/pipeline/08_resultados_hibrido_vs_lineas_base.ipynb`

## Notebook siguiente
- `notebooks/analysis/09_analisis_errores_hibrido.ipynb`


## Configuración de ejecución
Opcionalmente se puede fijar un barrido o freeze específicos. Si se dejan vacíos, se toma la corrida más reciente.

In [ ]:
BARRIDO_DIR = ''  # Ejemplo: data/outputs/barridos_hibridos/20260310_202656
FREEZE_DIR = ''   # Ejemplo: data/outputs/freeze_lexico_20260310_232420
TOP_BARRIDO = 30


In [ ]:
from pathlib import Path
import subprocess

cwd = Path.cwd().resolve()
candidatos = [cwd, *cwd.parents]
repo = next((p for p in candidatos if (p / 'scripts' / 'cerrar_modelos_dev.py').exists()), None)
if repo is None:
    raise RuntimeError('No se pudo resolver la raíz del repositorio desde el notebook actual.')

cmd = ['python', 'scripts/cerrar_modelos_dev.py', '--top-barrido', str(TOP_BARRIDO)]
if BARRIDO_DIR:
    cmd += ['--barrido-dir', BARRIDO_DIR]
if FREEZE_DIR:
    cmd += ['--freeze-dir', FREEZE_DIR]

print('$', ' '.join(cmd))
proc = subprocess.run(cmd, cwd=repo, text=True, capture_output=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError('Fallo en el cierre formal de modelos en dev.')

lineas = [ln.strip() for ln in proc.stdout.splitlines() if ln.strip()]
out_dir = Path(lineas[-1]) if lineas else None
print('Directorio generado:', out_dir)


In [ ]:
if out_dir and out_dir.exists():
    print('Artefactos generados:')
    for p in sorted(out_dir.glob('*')):
        print('-', p.name)
else:
    print('No se pudo resolver el directorio de salida.')
